In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Wage levels (ai x ict_spec)

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/data_panel/'

df_master = pd.read_csv(path + 'full_panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  tert_edu  spec_ict  \
0    AT       C      26.74  2021         9.61      31.8     28.74   
1    AT       C      26.07  2023        12.31      33.5     28.37   
2    AT       C      27.12  2025        32.54      36.2     30.02   
3    AT       F      22.43  2021         3.12      21.2      8.24   
4    AT       F      20.32  2023         4.28      22.5      9.03   
..   ..     ...        ...   ...          ...       ...       ...   
703  SK       M      11.53  2023        12.22      67.6     19.53   
704  SK       M      11.73  2025        28.48      64.8     23.11   
705  SK       N       7.35  2021         8.60      23.2     16.90   
706  SK       N       7.21  2023        10.16      25.0     13.37   
707  SK       N       7.37  2025        19.79      28.0     13.68   

     training_ict  unempl_r  infl_r      gdp  log_gdp  
0           20.37       6.2     2.8  45380.0    10.72  
1           25.60       5.1     7.7  52330.0    10.87  
2  

## panel

In [30]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [31]:
df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()

df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict'] = df_model['ai_adoption'] * df_model['spec_ict']


print(f"Observations: {len(df_model)}")
print(df_model.head())

Observations: 536
          geo nace_r2  real_wage  ai_adoption  spec_ict  training_ict  \
id   year                                                               
AT_C 2021  AT       C      26.74         9.61     28.56         22.98   
     2022  AT       C      26.18        10.96     28.37         25.60   
     2023  AT       C      26.07        12.31     29.20         25.83   
     2024  AT       C      27.05        22.71     30.02         26.06   
AT_F 2021  AT       F      22.43         3.12      8.64          8.28   

           productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
id   year                                                                      
AT_C 2021        102.25  55.82             36.58      31.8  111.46      4.63   
     2022        108.40  56.10             37.32      33.0  121.07      4.69   
     2023        108.93  57.52             38.71      33.5  130.40      4.69   
     2024        108.60  57.66             40.13      35.8  134.21    

In [4]:
# entity_effects=True - cross-sectional effect 
# time_effects=True - time fixed effect 
# without these variables will be just pooled OLS
# cluster_entity=True - calculates Clustered Standard Errors (treat the grouped observations (Austria-sector C) as a single cluster (group), not independently). Allow the errors within this group to be correlated

### ICT specialist 

In [32]:
y = df_model['log_real_wage']

exog_vars = [
    'ai_adoption',       # Main Effect: AI
    'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict',          # Interaction: AI * ICT
    # 'share_high_skill',  # Control: 
    'log_prod',          # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'tert_edu'           # Control: Education
]

X = df_model[exog_vars]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full)


                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.2100
Estimator:                   PanelOLS   R-squared (Between):              0.2639
No. Observations:                 536   R-squared (Within):              -0.0552
Date:                Sun, May 31 2026   R-squared (Overall):              0.2619
Time:                        12:28:16   Log-likelihood                    1029.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.413
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,393)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             3.7696
                            

In [33]:
from scipy import stats

beta_ai = res_panel_full.params['ai_adoption']
gamma = res_panel_full.params['ai_x_ict']

# Use clustered covariance matrix
vcov  = res_panel_full.cov
var_beta = vcov.loc['ai_adoption', 'ai_adoption']
var_gamma = vcov.loc['ai_x_ict', 'ai_x_ict']
cov_bg  = vcov.loc['ai_adoption', 'ai_x_ict']

z = df_model['spec_ict']
z_low, z_mean, z_high = np.percentile(z, [25, 50, 75])
dof         = res_panel_full.df_resid

def me_stats(z_val):
    me = beta_ai + gamma * z_val
    se = np.sqrt(var_beta + z_val**2 * var_gamma + 2 * z_val * cov_bg)
    t  = me / se
    p  = 2 * stats.t.sf(np.abs(t), df=dof)
    ci_lo = me - 1.96 * se
    ci_hi = me + 1.96 * se
    stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
    return me, se, t, p, ci_lo, ci_hi, stars

print("=== Marginal Effect of AI Adoption on log Real Wage ===")
print(f"{'ICT level':<30} {'ME':>8} {'SE':>8} {'t':>7} {'p':>8} {'95% CI':<22} {'Sig'}")
print("-" * 95)
for label, z_val in [("Low  spec_ict (P25)", z_low),
                     ("Mean spec_ict (P50)", z_mean),
                     ("High spec_ict (P75)", z_high)]:
    me, se, t, p, ci_lo, ci_hi, stars = me_stats(z_val)
    print(f"{label:<30} {me:>8.5f} {se:>8.5f} {t:>7.3f} {p:>8.4f}  [{ci_lo:.5f}, {ci_hi:.5f}]  {stars}")


=== Marginal Effect of AI Adoption on log Real Wage ===
ICT level                            ME       SE       t        p 95% CI                 Sig
-----------------------------------------------------------------------------------------------
Low  spec_ict (P25)            -0.00309  0.00131  -2.359   0.0188  [-0.00566, -0.00052]  **
Mean spec_ict (P50)            -0.00292  0.00123  -2.378   0.0179  [-0.00532, -0.00051]  **
High spec_ict (P75)            -0.00253  0.00105  -2.402   0.0168  [-0.00460, -0.00047]  **


In [8]:
# --- Extract from panel model (linearmodels syntax differs from statsmodels) ---
beta_ai   = res_panel_full.params['ai_adoption']
gamma     = res_panel_full.params['ai_x_ict']

# linearmodels uses .cov not .cov_params()
vcov      = res_panel_full.cov
var_beta  = vcov.loc['ai_adoption', 'ai_adoption']
var_gamma = vcov.loc['ai_x_ict', 'ai_x_ict']
cov_bg    = vcov.loc['ai_adoption', 'ai_x_ict']

# linearmodels uses df_resid not df_resid directly — check attribute
dof       = res_panel_full.df_resid

z         = df_model['spec_ict']

def me_stats(z_val, beta, gamma, var_b, var_g, cov_bg, dof):
    me    = beta + gamma * z_val
    se    = np.sqrt(var_b + z_val**2 * var_g + 2 * z_val * cov_bg)
    t     = me / se
    p     = 2 * stats.t.sf(np.abs(t), df=dof)
    ci_lo = me - 1.96 * se
    ci_hi = me + 1.96 * se
    stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
    return me, se, t, p, ci_lo, ci_hi, stars

# --- 1. Marginal Effect at the Mean (MEM) ---
z_mean = z.mean()
me, se, t, p, ci_lo, ci_hi, stars = me_stats(
    z_mean, beta_ai, gamma, var_beta, var_gamma, cov_bg, dof
)

print("=== Marginal Effect at the Mean (MEM) ===")
print(f"  spec_ict mean value : {z_mean:.4f}")
print(f"  Marginal effect     : {me:.5f} {stars}")
print(f"  Std. error          : {se:.5f}")
print(f"  t-statistic         : {t:.3f}")
print(f"  p-value             : {p:.4f}")
print(f"  95% CI              : [{ci_lo:.5f}, {ci_hi:.5f}]")

# --- 2. Average Marginal Effect (AME) ---
me_all = beta_ai + gamma * z  # individual MEs for every observation
z_bar  = z.mean()             # in linear models AME = MEM, z_bar used for SE

me_ame, se_ame, t_ame, p_ame, ci_lo_ame, ci_hi_ame, stars_ame = me_stats(
    z_bar, beta_ai, gamma, var_beta, var_gamma, cov_bg, dof
)

print("\n=== Average Marginal Effect (AME) ===")
print(f"  Mean of individual MEs : {me_all.mean():.5f} {stars_ame}")
print(f"  Std. error             : {se_ame:.5f}")
print(f"  t-statistic            : {t_ame:.3f}")
print(f"  p-value                : {p_ame:.4f}")
print(f"  95% CI                 : [{ci_lo_ame:.5f}, {ci_hi_ame:.5f}]")

print("\nSignificance: *** p<0.01  ** p<0.05  * p<0.1")

=== Marginal Effect at the Mean (MEM) ===
  spec_ict mean value : 26.5894
  Marginal effect     : -0.00261 **
  Std. error          : 0.00109
  t-statistic         : -2.400
  p-value             : 0.0169
  95% CI              : [-0.00475, -0.00048]

=== Average Marginal Effect (AME) ===
  Mean of individual MEs : -0.00261 **
  Std. error             : 0.00109
  t-statistic            : -2.400
  p-value                : 0.0169
  95% CI                 : [-0.00475, -0.00048]

Significance: *** p<0.01  ** p<0.05  * p<0.1


In [34]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# 1. PESARAN CD TEST
# ─────────────────────────────────────────────
# res_panel_full.resids returns a Series with MultiIndex (id, year)
resid = res_panel_full.resids

# Reshape to wide: rows = year, columns = entity id
resid_wide = resid.unstack(level='id')   # shape: T x N  (4 x 134)

N = resid_wide.shape[1]
T = resid_wide.shape[0]

cd_sum = 0.0
for i in range(N):
    for j in range(i + 1, N):
        pair = resid_wide.iloc[:, [i, j]].dropna()
        if len(pair) > 1:
            corr = pair.iloc[:, 0].corr(pair.iloc[:, 1])
            cd_sum += corr

CD = np.sqrt(2 * T / (N * (N - 1))) * cd_sum
p_cd = 2 * (1 - stats.norm.cdf(abs(CD)))

print("=== Pesaran CD Test (Cross-Sectional Dependence) ===")
print(f"  CD statistic : {CD:.4f}")
print(f"  p-value      : {p_cd:.4f}")
print(f"  H0: No cross-sectional dependence")
if p_cd < 0.05:
    print("  → REJECT H0: cross-sectional dependence present")
    print("    Consider switching to cov_type='kernel' (Driscoll-Kraay SEs)")
else:
    print("  → FAIL TO REJECT H0: no significant cross-sectional dependence")

=== Pesaran CD Test (Cross-Sectional Dependence) ===
  CD statistic : -0.4744
  p-value      : 0.6352
  H0: No cross-sectional dependence
  → FAIL TO REJECT H0: no significant cross-sectional dependence


In [35]:
# ─────────────────────────────────────────────
# 2. VIF ON WITHIN-DEMEANED DATA
# ─────────────────────────────────────────────
# FE estimator works on within-demeaned data (entity + time demeaned)
# VIF on raw panel data would be misleading — demean first

vif_vars = ['ai_adoption', 'spec_ict', 'ai_x_ict', 'log_prod', 'FSI', 'tert_edu']
X_raw = df_model[vif_vars].copy()

# Entity demean (subtract entity mean)
entity_means = X_raw.groupby(level='id').transform('mean')
# Time demean (subtract time mean)
time_means   = X_raw.groupby(level='year').transform('mean')
# Grand mean
grand_mean   = X_raw.mean()

# Two-way within transformation: x_it - x_i. - x_.t + x_..
X_demeaned = X_raw - entity_means - time_means + grand_mean

vif_df = pd.DataFrame({
    'Variable': vif_vars,
    'VIF': [variance_inflation_factor(X_demeaned.values, i)
            for i in range(len(vif_vars))]
}).sort_values('VIF', ascending=False)

print("\n=== VIF (on within-demeaned regressors) ===")
print(vif_df.to_string(index=False))
print("\nRule of thumb: VIF > 10 → high multicollinearity")
print("Note: ai_adoption, spec_ict, ai_x_ict will naturally show high VIF")
print("      due to the interaction term — this is expected, not a flaw.")


=== VIF (on within-demeaned regressors) ===
   Variable      VIF
   ai_x_ict 5.500457
ai_adoption 5.261304
   spec_ict 1.134480
   log_prod 1.027375
        FSI 1.024586
   tert_edu 1.013083

Rule of thumb: VIF > 10 → high multicollinearity
Note: ai_adoption, spec_ict, ai_x_ict will naturally show high VIF
      due to the interaction term — this is expected, not a flaw.


In [12]:
X = df_model[exog_vars]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res_panel_cs_FE = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_cs_FE)

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0705
Estimator:                   PanelOLS   R-squared (Between):             -0.0467
No. Observations:                 536   R-squared (Within):               0.0705
Date:                Sun, May 31 2026   R-squared (Overall):             -0.0460
Time:                        10:44:46   Log-likelihood                    918.65
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      5.0091
Entities:                         134   P-value                           0.0001
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             3.1410
                            

In [13]:
from linearmodels.panel import PooledOLS

# Pooled OLS does not use Entity Effects
# It treats every row as an independent observation
mod_pool = PooledOLS(y, X)
res_pool = mod_pool.fit(cov_type='clustered', cluster_entity=True)
print(res_pool)

                          PooledOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7517
Estimator:                  PooledOLS   R-squared (Between):              0.7949
No. Observations:                 536   R-squared (Within):              -6.2630
Date:                Sun, May 31 2026   R-squared (Overall):              0.7517
Time:                        10:44:48   Log-likelihood                   -92.951
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      266.88
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             73.291
                            

In [14]:
from linearmodels.panel import BetweenOLS, FirstDifferenceOLS, RandomEffects
import pandas as pd
import statsmodels.api as sm

mod_be = BetweenOLS(y, X)
res_be = mod_be.fit()
print(res_be)

                         BetweenOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.8163
Estimator:                 BetweenOLS   R-squared (Between):              0.8163
No. Observations:                 134   R-squared (Within):              -16.451
Date:                Sun, May 31 2026   R-squared (Overall):              0.7105
Time:                        10:44:51   Log-likelihood                   -2.6131
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      94.085
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,127)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             94.085
                            

In [15]:
# "Weighted average of Fixed Effects and Between Effects."
# Best model if your unobserved country traits are NOT correlated with AI.
mod_re = RandomEffects(y, X)
res_re = mod_re.fit()
print(res_re)


                        RandomEffects Estimation Summary                        
Dep. Variable:          log_real_wage   R-squared:                        0.2186
Estimator:              RandomEffects   R-squared (Between):              0.4377
No. Observations:                 536   R-squared (Within):              -0.1371
Date:                Sun, May 31 2026   R-squared (Overall):              0.4342
Time:                        10:44:54   Log-likelihood                    706.70
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      24.662
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             24.662
                            

In [16]:
# Similar to Fixed Effects, often handles trends better.
X_fd = X.drop(columns=['const'])
mod_fd = FirstDifferenceOLS(y, X_fd)
res_fd = mod_fd.fit()
print(res_fd)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:          log_real_wage   R-squared:                        0.0990
Estimator:         FirstDifferenceOLS   R-squared (Between):             -0.1061
No. Observations:                 402   R-squared (Within):               0.0204
Date:                Sun, May 31 2026   R-squared (Overall):             -0.1060
Time:                        10:44:56   Log-likelihood                    613.51
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      7.2515
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             7.2515
                            

In [17]:
from linearmodels.panel import compare

comparison = {
    'Pooled OLS': res_pool,
    'Between': res_be,
    'Random Effects': res_re,
    'Fixed Effects full': res_panel_full, 
    'Fixed Effect cross-sect' : res_panel_cs_FE,
    'First Diff': res_fd
}

summary_table = compare(comparison)

print(summary_table)

                                                                Model Comparison                                                               
                               Pooled OLS           Between    Random Effects Fixed Effects full Fixed Effect cross-sect             First Diff
-----------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable               log_real_wage     log_real_wage     log_real_wage      log_real_wage           log_real_wage          log_real_wage
Estimator                       PooledOLS        BetweenOLS     RandomEffects           PanelOLS                PanelOLS     FirstDifferenceOLS
No. Observations                      536               134               536                536                     536                    402
Cov. Est.                       Clustered        Unadjusted        Unadjusted          Clustered               Clustered             Una

In [9]:
import statsmodels.stats.diagnostic as smd
from scipy import stats

# Model selection test (WALD TEST)
# https://bashtage.github.io/linearmodels/panel/panel/linearmodels.panel.results.PanelEffectsResults.f_pooled.html
# 1. F-Test for Fixed Effects (Pooled OLS vs. Fixed Effects)
# H0: Pooled OLS is better (Entity effects are zero)
# H1: Fixed Effects is bette

print(f"1. F-Test for Entity Effects (Pooled vs FE):")
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")



1. F-Test for Entity Effects (Pooled vs FE):
   F-Stat: 187.5317
   P-Value: 0.0000
   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.


In [18]:
# 2. Breusch-Pagan LM Test (Pooled OLS vs. Random Effects)
# H0: Variance of random effects is 0 (Pooled OLS is fine)
# H1: Variance > 0 (Random Effects are needed)
# We calculate this manually using residuals from Pooled OLS
resid_pool = res_pool.resids
n = len(df_model.index.get_level_values(0).unique()) # Entities
T = len(df_model.index.get_level_values(1).unique()) # Time periods
# Calculation
lm_stat = (n * T) / (2 * (T - 1)) * (
    (resid_pool.groupby(level=0).sum() ** 2).sum() / (resid_pool ** 2).sum() - 1
) ** 2
lm_pval = 1 - stats.chi2.cdf(lm_stat, df=1)

print(f"\n2. Breusch-Pagan LM Test (Pooled vs Random Effects):")
print(f"   LM Stat: {lm_stat:.4f}")
print(f"   P-Value: {lm_pval:.4f}")
if lm_pval < 0.05:
    print("   -> Result: REJECT H0. Random Effects is better than Pooled.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")


2. Breusch-Pagan LM Test (Pooled vs Random Effects):
   LM Stat: 465.5256
   P-Value: 0.0000
   -> Result: REJECT H0. Random Effects is better than Pooled.


In [20]:
# 3. Hausman Test (Fixed Effects vs. Random Effects)
# H0: Random Effects is consistent (Use RE - it's more efficient)
# H1: Random Effects is biased (Use FE - it's safer)
b_fe = res_panel_full.params
b_re = res_re.params
cov_fe = res_panel_full.cov
cov_re = res_re.cov
# Calculate Chi-Square
diff = b_fe - b_re
# Note: Usually we drop the constant for Hausman as FE doesn't estimate it the same way
diff = diff.drop('const')
cov_diff = cov_fe.loc[diff.index, diff.index] - cov_re.loc[diff.index, diff.index]
hausman_stat = diff.dot(np.linalg.inv(cov_diff)).dot(diff)
hausman_pval = 1 - stats.chi2.cdf(hausman_stat, df=len(diff))

print(f"\n3. Hausman Test (FE vs RE):")
print(f"   Chi2 Stat: {hausman_stat:.4f}")
print(f"   P-Value: {hausman_pval:.4f}")
if hausman_pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects (RE is biased).")
else:
    print("   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).")


3. Hausman Test (FE vs RE):
   Chi2 Stat: 5.1542
   P-Value: 0.5242
   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).


In [21]:
# 4. Heteroskedasticity Tests (White & Breusch-Pagan)
# We run these on the POOLED residuals (standard approach)
# H0: Homoskedasticity (Variance is constant) -> Good
# H1: Heteroskedasticity (Variance changes) -> Bad (Need Robust Errors)
bp_test = smd.het_breuschpagan(res_panel_full.resids, res_panel_full.model.exog.dataframe)
white_test = smd.het_white(res_panel_full.resids, res_panel_full.model.exog.dataframe)

print(f"4. Heteroskedasticity Tests:")
print(f"   Breusch-Pagan P-Value: {bp_test[1]:.4f}")
print(f"   White Test P-Value:    {white_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')")
else:
    print("   -> Result: Homoskedasticity. (Data is clean).")

4. Heteroskedasticity Tests:
   Breusch-Pagan P-Value: 0.0395
   White Test P-Value:    0.0000
   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')


In [ ]:
# # 5. Serial Correlation (Breusch-Godfrey)
# # H0: No Serial Correlation
# # H1: Serial Correlation exists
# # Note: We restrict lags to 1 because you only have 3 years of data
# bg_test = smd.acorr_breusch_godfrey(res_panel_full, nlags=1)

# print(f"\n5. Serial Correlation (Breusch-Godfrey):")
# print(f"   P-Value: {bg_test[1]:.4f}")
# if bg_test[1] < 0.05:
#     print("   -> Result: SERIAL CORRELATION DETECTED. (Use Clustered Errors).")
# else:
#     print("   -> Result: No Serial Correlation.")

In [22]:
# 6. Chow Test (Test for Poolability)
# This is automatically calculated in the PanelOLS summary!
# It tests if the slopes are different for every entity.
# linearmodels calls this "F-test for Poolability"
print(f"6. Chow Test (Poolability):")
# We access the F-statistic directly from the results object
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).")


6. Chow Test (Poolability):
   F-Stat: 187.5317
   P-Value: 0.0000
   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).


In [23]:
# 7. Wald Test (Joint Significance)
# Example: Does 'training_ict' AND 'share_high_skill' jointly equal zero?
# Use this to test if your Control Variables matter as a group.
formula = 'spec_ict = 0, ai_adoption = 0, ai_x_ict = 0 '
# Note: If you removed these variables, change the formula to 'spec_ict = 0, ai_adoption = 0'
try:
    wald_res = res_panel_full.wald_test(formula=formula)
    print(f"\n7. Wald Test (Joint Significance of Controls):")
    print(f"   Stat: {wald_res.stat:.4f}, P-Value: {wald_res.pval:.4f}")
except:
    print("\n7. Wald Test: Skipped (Variables not in model).")


7. Wald Test (Joint Significance of Controls):
   Stat: 5.8519, P-Value: 0.1190


In [24]:
# Calculate the variance
variance_data = df_model['ai_adoption']
mean_total = variance_data.mean()

# Between Variance (Variation across countries)
between_var = df_model.groupby('id')['ai_adoption'].mean().var()

# Within Variance (Variation over time within a country)
within_var = (df_model['ai_adoption'] - df_model.groupby('id')['ai_adoption'].transform('mean')).var()

print(f"Variance BETWEEN Sectors (Structure): {between_var:.4f}")
print(f"Variance WITHIN Sectors (Time):       {within_var:.4f}")

ratio = between_var / within_var
print(f"Ratio (Between / Within):             {ratio:.2f}")

Variance BETWEEN Sectors (Structure): 115.2127
Variance WITHIN Sectors (Time):       20.6323
Ratio (Between / Within):             5.58


## Wage levels (ai x ict_trainings)

In [36]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [37]:

df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()
df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict_tr'] = df_model['ai_adoption'] * df_model['training_ict']

print(f"Observations: {len(df_model)}")
print(df_model.head())

Observations: 536
          geo nace_r2  real_wage  ai_adoption  spec_ict  training_ict  \
id   year                                                               
AT_C 2021  AT       C      26.74         9.61     28.56         22.98   
     2022  AT       C      26.18        10.96     28.37         25.60   
     2023  AT       C      26.07        12.31     29.20         25.83   
     2024  AT       C      27.05        22.71     30.02         26.06   
AT_F 2021  AT       F      22.43         3.12      8.64          8.28   

           productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
id   year                                                                      
AT_C 2021        102.25  55.82             36.58      31.8  111.46      4.63   
     2022        108.40  56.10             37.32      33.0  121.07      4.69   
     2023        108.93  57.52             38.71      33.5  130.40      4.69   
     2024        108.60  57.66             40.13      35.8  134.21    

In [38]:
y = df_model['log_real_wage']
exog_vars = [
    'ai_adoption',       # Main Effect: AI
    # 'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict_tr',          # Interaction: AI * ICT Training
    'training_ict',      # Control: ICT Training
    'log_prod',      # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'share_high_skill'   # Control: Human Capital
]
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full)


                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.2461
Estimator:                   PanelOLS   R-squared (Between):              0.4233
No. Observations:                 536   R-squared (Within):              -0.0436
Date:                Sun, May 31 2026   R-squared (Overall):              0.4204
Time:                        12:29:08   Log-likelihood                    1042.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.381
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,393)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             7.1724
                            

In [40]:
# ─────────────────────────────────────────────
# 2. VIF ON WITHIN-DEMEANED DATA
# ─────────────────────────────────────────────
# FE estimator works on within-demeaned data (entity + time demeaned)
# VIF on raw panel data would be misleading — demean first

vif_vars = ['ai_adoption', 'training_ict', 'ai_x_ict_tr', 'log_prod', 'FSI', 'share_high_skill']
X_raw = df_model[vif_vars].copy()

# Entity demean (subtract entity mean)
entity_means = X_raw.groupby(level='id').transform('mean')
# Time demean (subtract time mean)
time_means   = X_raw.groupby(level='year').transform('mean')
# Grand mean
grand_mean   = X_raw.mean()

# Two-way within transformation: x_it - x_i. - x_.t + x_..
X_demeaned = X_raw - entity_means - time_means + grand_mean

vif_df = pd.DataFrame({
    'Variable': vif_vars,
    'VIF': [variance_inflation_factor(X_demeaned.values, i)
            for i in range(len(vif_vars))]
}).sort_values('VIF', ascending=False)

print("\n=== VIF (on within-demeaned regressors) ===")
print(vif_df.to_string(index=False))
print("\nRule of thumb: VIF > 10 → high multicollinearity")
print("Note: ai_adoption, training_ict, ai_x_ict_tr will naturally show high VIF")
print("      due to the interaction term — this is expected, not a flaw.")


=== VIF (on within-demeaned regressors) ===
        Variable      VIF
     ai_x_ict_tr 6.730650
     ai_adoption 6.532723
    training_ict 1.157276
        log_prod 1.041647
share_high_skill 1.027281
             FSI 1.022843

Rule of thumb: VIF > 10 → high multicollinearity
Note: ai_adoption, training_ict, ai_x_ict_tr will naturally show high VIF
      due to the interaction term — this is expected, not a flaw.


In [39]:
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True)
res_panel_cs_FE = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_cs_FE)

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0971
Estimator:                   PanelOLS   R-squared (Between):              0.1944
No. Observations:                 536   R-squared (Within):               0.0971
Date:                Sun, May 31 2026   R-squared (Overall):              0.1938
Time:                        12:29:43   Log-likelihood                    926.41
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.0968
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             5.3135
                            

In [25]:
# Pooled OLS does not use Entity Effects
# It treats every row as an independent observation
mod_pool = PooledOLS(y, X)
res_pool = mod_pool.fit(cov_type='clustered', cluster_entity=True)
print(res_pool)

                          PooledOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7426
Estimator:                  PooledOLS   R-squared (Between):              0.7752
No. Observations:                 536   R-squared (Within):              -4.5528
Date:                Thu, Mar 12 2026   R-squared (Overall):              0.7426
Time:                        00:30:48   Log-likelihood                   -102.60
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      254.32
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             44.578
                            

In [26]:
mod_be = BetweenOLS(y, X)
res_be = mod_be.fit()
print(res_be)

                         BetweenOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7902
Estimator:                 BetweenOLS   R-squared (Between):              0.7902
No. Observations:                 134   R-squared (Within):              -12.438
Date:                Thu, Mar 12 2026   R-squared (Overall):              0.7091
Time:                        00:30:50   Log-likelihood                   -11.531
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      79.722
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,127)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             79.722
                            

In [27]:
# "Weighted average of Fixed Effects and Between Effects."
# Best model if your unobserved country traits are NOT correlated with AI.
mod_re = RandomEffects(y, X)
res_re = mod_re.fit()
print(res_re)

                        RandomEffects Estimation Summary                        
Dep. Variable:          log_real_wage   R-squared:                        0.2405
Estimator:              RandomEffects   R-squared (Between):              0.4333
No. Observations:                 536   R-squared (Within):              -0.0256
Date:                Thu, Mar 12 2026   R-squared (Overall):              0.4305
Time:                        00:30:53   Log-likelihood                    740.36
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      27.916
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             27.916
                            

In [30]:
# Similar to Fixed Effects, often handles trends better.
X_fd = X.drop(columns=['const'])
mod_fd = FirstDifferenceOLS(y, X_fd)
res_fd = mod_fd.fit()
print(res_fd)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:          log_real_wage   R-squared:                        0.1139
Estimator:         FirstDifferenceOLS   R-squared (Between):              0.0724
No. Observations:                 402   R-squared (Within):               0.0511
Date:                Thu, Mar 12 2026   R-squared (Overall):              0.0724
Time:                        00:31:18   Log-likelihood                    616.86
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      8.4821
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             8.4821
                            

In [ ]:
comparison = {
    'Pooled OLS': res_pool,
    'Between': res_be,
    'Random Effects': res_re,
    'Fixed Effects full': res_panel_full, 
    'Fixed Effect cross-sect' : res_panel_cs_FE,
    'First Diff': res_fd
}
summary_table = compare(comparison)
print(summary_table)

                                                                Model Comparison                                                               
                               Pooled OLS           Between    Random Effects Fixed Effects full Fixed Effect cross-sect             First Diff
-----------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable               log_real_wage     log_real_wage     log_real_wage      log_real_wage           log_real_wage          log_real_wage
Estimator                       PooledOLS        BetweenOLS     RandomEffects           PanelOLS                PanelOLS     FirstDifferenceOLS
No. Observations                      536               134               536                536                     536                    402
Cov. Est.                       Clustered        Unadjusted        Unadjusted          Clustered               Clustered             Una

In [32]:
# Model selection test (WALD TEST)
# https://bashtage.github.io/linearmodels/panel/panel/linearmodels.panel.results.PanelEffectsResults.f_pooled.html
# 1. F-Test for Fixed Effects (Pooled OLS vs. Fixed Effects)
# H0: Pooled OLS is better (Entity effects are zero)
# H1: Fixed Effects is bette

print(f"1. F-Test for Entity Effects (Pooled vs FE):")
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")



1. F-Test for Entity Effects (Pooled vs FE):
   F-Stat: 203.9620
   P-Value: 0.0000
   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.


In [33]:
# 2. Breusch-Pagan LM Test (Pooled OLS vs. Random Effects)
# H0: Variance of random effects is 0 (Pooled OLS is fine)
# H1: Variance > 0 (Random Effects are needed)
# We calculate this manually using residuals from Pooled OLS
resid_pool = res_pool.resids
n = len(df_model.index.get_level_values(0).unique()) # Entities
T = len(df_model.index.get_level_values(1).unique()) # Time periods
# Calculation
lm_stat = (n * T) / (2 * (T - 1)) * (
    (resid_pool.groupby(level=0).sum() ** 2).sum() / (resid_pool ** 2).sum() - 1
) ** 2
lm_pval = 1 - stats.chi2.cdf(lm_stat, df=1)

print(f"\n2. Breusch-Pagan LM Test (Pooled vs Random Effects):")
print(f"   LM Stat: {lm_stat:.4f}")
print(f"   P-Value: {lm_pval:.4f}")
if lm_pval < 0.05:
    print("   -> Result: REJECT H0. Random Effects is better than Pooled.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")


2. Breusch-Pagan LM Test (Pooled vs Random Effects):
   LM Stat: 545.4818
   P-Value: 0.0000
   -> Result: REJECT H0. Random Effects is better than Pooled.


In [34]:
# 3. Hausman Test (Fixed Effects vs. Random Effects)
# H0: Random Effects is consistent (Use RE - it's more efficient)
# H1: Random Effects is biased (Use FE - it's safer)
b_fe = res_panel_full.params
b_re = res_re.params
cov_fe = res_panel_full.cov
cov_re = res_re.cov
# Calculate Chi-Square
diff = b_fe - b_re
# Note: Usually we drop the constant for Hausman as FE doesn't estimate it the same way
diff = diff.drop('const')
cov_diff = cov_fe.loc[diff.index, diff.index] - cov_re.loc[diff.index, diff.index]
hausman_stat = diff.dot(np.linalg.inv(cov_diff)).dot(diff)
hausman_pval = 1 - stats.chi2.cdf(hausman_stat, df=len(diff))

print(f"\n3. Hausman Test (FE vs RE):")
print(f"   Chi2 Stat: {hausman_stat:.4f}")
print(f"   P-Value: {hausman_pval:.4f}")
if hausman_pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects (RE is biased).")
else:
    print("   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).")


3. Hausman Test (FE vs RE):
   Chi2 Stat: 7.2937
   P-Value: 0.2945
   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).


In [35]:
# 4. Heteroskedasticity Tests (White & Breusch-Pagan)
# We run these on the POOLED residuals (standard approach)
# H0: Homoskedasticity (Variance is constant) -> Good
# H1: Heteroskedasticity (Variance changes) -> Bad (Need Robust Errors)
bp_test = smd.het_breuschpagan(res_panel_full.resids, res_panel_full.model.exog.dataframe)
white_test = smd.het_white(res_panel_full.resids, res_panel_full.model.exog.dataframe)

print(f"4. Heteroskedasticity Tests:")
print(f"   Breusch-Pagan P-Value: {bp_test[1]:.4f}")
print(f"   White Test P-Value:    {white_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')")
else:
    print("   -> Result: Homoskedasticity. (Data is clean).")

4. Heteroskedasticity Tests:
   Breusch-Pagan P-Value: 0.0546
   White Test P-Value:    0.0000
   -> Result: Homoskedasticity. (Data is clean).


In [36]:
# 6. Chow Test (Test for Poolability)
# This is automatically calculated in the PanelOLS summary!
# It tests if the slopes are different for every entity.
# linearmodels calls this "F-test for Poolability"
print(f"6. Chow Test (Poolability):")
# We access the F-statistic directly from the results object
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).")


6. Chow Test (Poolability):
   F-Stat: 203.9620
   P-Value: 0.0000
   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).


In [39]:
# 7. Wald Test (Joint Significance)
# Example: Does 'training_ict' AND 'share_high_skill' jointly equal zero?
# Use this to test if your Control Variables matter as a group.
formula = 'training_ict = 0, ai_adoption = 0, ai_x_ict_tr = 0'
# Note: If you removed these variables, change the formula to 'spec_ict = 0, ai_adoption = 0'
try:
    wald_res = res_panel_full.wald_test(formula=formula)
    print(f"\n7. Wald Test (Joint Significance of Controls):")
    print(f"   Stat: {wald_res.stat:.4f}, P-Value: {wald_res.pval:.4f}")
except:
    print("\n7. Wald Test: Skipped (Variables not in model).")


7. Wald Test (Joint Significance of Controls):
   Stat: 9.0651, P-Value: 0.0284


In [40]:
# Calculate the variance
variance_data = df_model['ai_adoption']
mean_total = variance_data.mean()

# Between Variance (Variation across countries)
between_var = df_model.groupby('id')['ai_adoption'].mean().var()

# Within Variance (Variation over time within a country)
within_var = (df_model['ai_adoption'] - df_model.groupby('id')['ai_adoption'].transform('mean')).var()

print(f"Variance BETWEEN Sectors (Structure): {between_var:.4f}")
print(f"Variance WITHIN Sectors (Time):       {within_var:.4f}")

ratio = between_var / within_var
print(f"Ratio (Between / Within):             {ratio:.2f}")

Variance BETWEEN Sectors (Structure): 115.2127
Variance WITHIN Sectors (Time):       20.6323
Ratio (Between / Within):             5.58


## Run models without productivity

In [7]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)

df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()
df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict'] = df_model['ai_adoption'] * df_model['spec_ict']

print(f"Observations: {len(df_model)}")
print(df_model.head())

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [8]:
y = df_model['log_real_wage']
exog_vars = [
    'ai_adoption',       # Main Effect: AI
    'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict',          # Interaction: AI * ICT
    # 'training_ict',      # Control: ICT Training
    # 'log_prod',      # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'share_high_skill'   # Control: Human Capital
]
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full_test1 = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full_test1)

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0526
Estimator:                   PanelOLS   R-squared (Between):             -0.0376
No. Observations:                 536   R-squared (Within):              -0.0344
Date:                Tue, Apr 14 2026   R-squared (Overall):             -0.0376
Time:                        10:45:01   Log-likelihood                    980.76
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.3744
Entities:                         134   P-value                           0.0007
Avg Obs:                       4.0000   Distribution:                   F(5,394)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             2.2544
                            

In [9]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)


df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()
df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict_tr'] = df_model['ai_adoption'] * df_model['training_ict']
print(f"Observations: {len(df_model)}")
print(df_model.head())

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [10]:
y = df_model['log_real_wage']
exog_vars = [
    'ai_adoption',       # Main Effect: AI
    # 'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict_tr',          # Interaction: AI * ICT Training
    'training_ict',      # Control: ICT Training
    # 'log_prod',      # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'share_high_skill'   # Control: Human Capital
]
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full_test2 = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full_test2)



                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0907
Estimator:                   PanelOLS   R-squared (Between):              0.1400
No. Observations:                 536   R-squared (Within):               0.0421
Date:                Tue, Apr 14 2026   R-squared (Overall):              0.1394
Time:                        10:45:17   Log-likelihood                    991.75
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.8559
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(5,394)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             2.8982
                            

In [11]:
comparison = {
    'Fixed Effects full spec': res_panel_full_test1, 
    'Fixed Effects full train': res_panel_full_test2, 

}
summary_table = compare(comparison)
print(summary_table)

NameError: name 'compare' is not defined

## CRE (Correlated Random Effect model)

In [15]:
df_master = pd.read_csv(path + 'full_panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  log_wage  ai_adoption  tert_edu  \
0    AT       C      26.74  2021      3.29         9.61      31.8   
1    AT       C      26.07  2023      3.26        12.31      33.5   
2    AT       F      22.43  2021      3.11         3.12      21.2   
3    AT       F      20.32  2023      3.01         4.28      22.5   
4    AT       G      21.35  2021      3.06         6.94      23.1   
..   ..     ...        ...   ...       ...          ...       ...   
508  SK       J      14.55  2023      2.68        21.62      72.6   
509  SK       M      11.84  2021      2.47         3.67      65.6   
510  SK       M      11.53  2023      2.44        12.22      67.6   
511  SK       N       7.35  2021      1.99         8.60      23.2   
512  SK       N       7.21  2023      1.98        10.16      25.0   

     productivity  log_prod  spec_ict  training_ict  unempl_r  infl_r  \
0          102.25      4.63     28.74         20.37       6.2     2.8   
1          108.93      4.

In [16]:
print(df_master.columns)

Index(['geo', 'nace_r2', 'real_wage', 'year', 'log_wage', 'ai_adoption',
       'tert_edu', 'productivity', 'log_prod', 'spec_ict', 'training_ict',
       'unempl_r', 'infl_r', 'gdp', 'log_gdp'],
      dtype='object')


In [17]:
exec(open("cre_mundlak_model_v2.py").read())
results = run_full_analysis(df_master)

CRE/Mundlak script loaded. Run: results = run_full_analysis(df_master)

####################################################################
  SPECIFICATION A — ICT SPECIALISTS + tert_edu
####################################################################

  CRE / MUNDLAK MODEL — ICT SPECIALISTS

  Variable                                Coef         SE     p-val  Sig
  ------------------------------------------------------------------
  ai_adoption                         -0.00054    0.00386    0.8897  
  spec_ict                             0.00137    0.00173    0.4289  
  ai_x_spec_ict                        0.00001    0.00004    0.8924  
  --- Mundlak means (BETWEEN effects) ---
  ai_adoption_mean                     0.04667    0.00694    0.0000  ***
  spec_ict_mean                        0.00009    0.00325    0.9770  
  ai_x_spec_ict_mean                  -0.00043    0.00010    0.0000  ***
  log_prod                             0.60807    0.04854    0.0000  ***
  tert_edu        

In [19]:
# 1. Load the robustness script
exec(open("cre_robustness_checks.py").read())

# 2. Prepare the df with Mundlak means (if not already done)
df = prepare_panel(df_master)
df = add_mundlak_means(df, ['ai_adoption', 'spec_ict', 'training_ict',
                             'ai_x_spec_ict', 'ai_x_training_ict',
                            #  'log_prod', 'FSI'])
                            'log_prod'])

# 3. Run robustness checks
rc = run_robustness(df, results['spec_spec_ict'], results['spec_training_ict'])

Robustness script loaded.
Call: rc = run_robustness(df, results['spec_spec_ict'], results['spec_training_ict'])

####################################################################
  ROBUSTNESS CHECK 1 — CRE WITH MACRO TIME CONTROLS
####################################################################


PatsyError: Error evaluating factor: NameError: name 'FSI' is not defined
    log_wage ~ ai_adoption + spec_ict + ai_x_spec_ict + ai_adoption_mean + spec_ict_mean + ai_x_spec_ict_mean + log_prod + FSI + tert_edu + unempl_r + infl_r + C(nace_r2)
                                                                                                                           ^^^

In [20]:
df_master = pd.read_csv(path + 'full_panel_master.csv')
print(df_master.head())

  geo nace_r2  real_wage  year  log_wage  ai_adoption  tert_edu  productivity  \
0  AT       C      26.74  2021      3.29         9.61      31.8        102.25   
1  AT       C      26.07  2023      3.26        12.31      33.5        108.93   
2  AT       F      22.43  2021      3.11         3.12      21.2         75.26   
3  AT       F      20.32  2023      3.01         4.28      22.5         86.76   
4  AT       G      21.35  2021      3.06         6.94      23.1         66.47   

   log_prod  spec_ict  training_ict  unempl_r  infl_r      gdp  log_gdp  
0      4.63     28.74         20.37       6.2     2.8  45380.0    10.72  
1      4.69     28.37         25.60       5.1     7.7  52330.0    10.87  
2      4.32      8.24          9.65       6.2     2.8  45380.0    10.72  
3      4.46      9.03          6.91       5.1     7.7  52330.0    10.87  
4      4.20     16.25         17.43       6.2     2.8  45380.0    10.72  


In [21]:
exec(open("cre_mundlak_additive.py").read())

add_results = run_full_analysis_additive(df_master)

CRE/Mundlak ADDITIVE script loaded. Run: add_results = run_full_analysis_additive(df_master)


KeyError: "Columns not found: 'FSI'"

In [13]:
exec(open("cre_robustness_additive.py").read())

# For additive robustness, you only need means of ai, ICT, log_prod, FSI
df_add = prepare_panel(df_master)
df_add = add_mundlak_means(df_add, ['ai_adoption', 'spec_ict', 'training_ict',
                                    'log_prod', 'FSI'])

add_rc = run_robustness_additive(
    df_add,
    add_results['add_spec_spec_ict'],
    add_results['add_spec_training_ict']
)

Additive robustness script loaded.
Call: add_rc = run_robustness_additive(df, add_results['add_spec_spec_ict'], add_results['add_spec_training_ict'])

####################################################################
  ROBUSTNESS (ADDITIVE) — CRE WITH MACRO TIME CONTROLS
####################################################################

####################################################################
  ROBUSTNESS (ADDITIVE) — TWO-WAY FE (linearmodels)
####################################################################
  Two-Way FE (additive) estimated successfully.

####################################################################
  COMPARISON TABLES + MEM/AME + VIF (ADDITIVE)
####################################################################

  COEFFICIENT COMPARISON (ADDITIVE) — SPEC_ICT  |  edu: tert_edu
  Col 1: CRE + year dummies | Col 2: CRE + macro controls | Col 3: Two-Way FE
  Variable                               CRE+Year      CRE+Macro     Two-Way FE
  -----